# Experiment 3: Central tendency and variability

**Roll number:** 16014324022  
**Aim:** Compute and interpret frequency, mean, median, mode, range, variance, standard deviation, and interquartile range for single attributes and grouped data.

## 1. Load and inspect the data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Run the notebook from this folder so the local CSV can be found.
notebook_dir = Path.cwd()
data_path = notebook_dir / "irrigation_prediction.csv"
if not data_path.exists():
    raise FileNotFoundError(f"Place irrigation_prediction.csv beside this notebook: {data_path}")

df = pd.read_csv(data_path)
required = ["Temperature_C", "Humidity", "Sunlight_Hours", "Soil_Moisture", "Rainfall_mm"]
missing = [name for name in required if name not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Dataset: {data_path.name}; rows: {len(df):,}; columns: {len(df.columns)}")
print("Missing values in selected columns:")
print(df[required].isna().sum().to_string())
print("\nFirst five records:")
print(df[required].head().to_string(index=False))


Dataset: irrigation_prediction.csv; rows: 10,000; columns: 20
Missing values in selected columns:
Temperature_C     0
Humidity          0
Sunlight_Hours    0
Soil_Moisture     0
Rainfall_mm       0

First five records:
 Temperature_C  Humidity  Sunlight_Hours  Soil_Moisture  Rainfall_mm
         21.90     31.19            4.01          36.48      1167.70
         36.50     26.01           10.72          50.56       831.28
         41.83     76.41            7.75          40.07      1844.45
         37.22     43.32            8.90          12.75       306.26
         22.38     86.44           10.39          18.58      1875.63


## 2. Manual procedure without built-in statistical functions

The functions below use loops for counting, summation, mode frequency, and squared deviations. The small sample uses bubble sort for its median and quartiles. For quartiles, the method is the median of each half of the sorted sample. This may differ slightly from Pandas' interpolation rule. Sample variance divides by `n - 1`.


In [2]:
def manual_count(values):
    count = 0
    for _ in values:
        count += 1
    return count

def manual_mean(values):
    total = 0.0
    for value in values:
        total += value
    return total / manual_count(values)

def bubble_sort(values):
    ordered = list(values)
    n = manual_count(ordered)
    for i in range(n):
        swapped = False
        for j in range(n - i - 1):
            if ordered[j] > ordered[j + 1]:
                ordered[j], ordered[j + 1] = ordered[j + 1], ordered[j]
                swapped = True
        if not swapped:
            break
    return ordered

def middle_of_sorted(values):
    n = manual_count(values)
    if n % 2:
        return values[n // 2]
    return (values[n // 2 - 1] + values[n // 2]) / 2

def manual_stats(values):
    n = manual_count(values)
    if n < 2:
        raise ValueError("At least two observations are required for sample variance")
    ordered = bubble_sort(values)
    mean = manual_mean(values)
    frequencies = {}
    for value in values:
        frequencies[value] = frequencies.get(value, 0) + 1
    highest_frequency = 0
    modes = []
    for value, frequency in frequencies.items():
        if frequency > highest_frequency:
            highest_frequency, modes = frequency, [value]
        elif frequency == highest_frequency:
            modes.append(value)
    squared_deviation = 0.0
    for value in values:
        squared_deviation += (value - mean) ** 2
    variance = squared_deviation / (n - 1)
    lower_half = ordered[:n // 2]
    upper_half = ordered[(n + 1) // 2:]
    q1, q3 = middle_of_sorted(lower_half), middle_of_sorted(upper_half)
    return {"count": n, "mean": mean, "median": middle_of_sorted(ordered),
            "modes": modes, "mode_frequency": highest_frequency,
            "range": ordered[-1] - ordered[0], "variance": variance,
            "std_dev": variance ** 0.5, "Q1": q1, "Q3": q3, "IQR": q3 - q1}

sample = df["Temperature_C"].dropna().head(20).tolist()
manual = manual_stats(sample)
print("First 20 nonmissing temperatures (°C):", sample)
for key, value in manual.items():
    print(f"{key}: {value}")
assert np.isclose(manual["mean"], pd.Series(sample).mean())
assert np.isclose(manual["variance"], pd.Series(sample).var(ddof=1))
assert np.isclose(manual["median"], pd.Series(sample).median())


First 20 nonmissing temperatures (°C): [21.9, 36.5, 41.83, 37.22, 22.38, 33.34, 28.02, 35.6, 16.59, 31.14, 35.21, 34.84, 40.91, 13.75, 22.37, 26.3, 21.29, 21.29, 17.28, 36.39]
count: 20
mean: 28.707499999999992
median: 29.58
modes: [21.29]
mode_frequency: 2
range: 28.08
variance: 74.32000921052634
std_dev: 8.6209053590981
Q1: 21.595
Q3: 35.995000000000005
IQR: 14.400000000000006


## 3. Single and multi-attribute analysis




In [3]:
def descriptive_row(series):
    clean = pd.to_numeric(series, errors="coerce").dropna()
    modes = clean.mode().tolist()
    return pd.Series({"frequency": len(clean), "mean": clean.mean(),
                      "median": clean.median(), "mode": modes,
                      "mode_frequency": clean.value_counts().iloc[0],
                      "range": clean.max() - clean.min(),
                      "variance": clean.var(ddof=1), "std_dev": clean.std(ddof=1),
                      "Q1": clean.quantile(0.25), "Q3": clean.quantile(0.75),
                      "IQR": clean.quantile(0.75) - clean.quantile(0.25)})

attributes = ["Temperature_C", "Humidity", "Sunlight_Hours", "Soil_Moisture", "Rainfall_mm"]
summary = pd.DataFrame({name: descriptive_row(df[name]) for name in attributes}).T
print(summary.to_string())
print("\nCategorical frequency and mode: Irrigation_Need")
print(df["Irrigation_Need"].value_counts(dropna=False).to_string())
print("Mode:", df["Irrigation_Need"].mode().tolist())


               frequency        mean    median                               mode mode_frequency    range       variance     std_dev       Q1        Q3      IQR
Temperature_C      10000   26.991423     27.09                            [32.74]             12     30.0      75.066186    8.664074    19.46      34.5    15.04
Humidity           10000   60.080339     60.04                            [48.24]              8     70.0     407.554269   20.187973   42.855    77.705    34.85
Sunlight_Hours     10000    7.518538      7.56                            [10.26]             27      7.0       4.064566    2.016077     5.76      9.26      3.5
Soil_Moisture      10000   36.969207     37.24                            [52.38]              9     57.0     269.972654   16.430845    22.86     50.94    28.08
Rainfall_mm        10000  1252.49942  1250.335  [48.09, 226.24, 1061.21, 1361.72]              3  2499.31  512057.886408  715.582201  634.155  1880.265  1246.11

Categorical frequency and mode: I

## 4. Temperature quartiles and outliers

In [4]:
temperature = df["Temperature_C"].dropna()
q1 = temperature.quantile(0.25)
median = temperature.median()
q3 = temperature.quantile(0.75)
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = temperature[(temperature < lower_fence) | (temperature > upper_fence)]
print(f"Q1={q1:.2f} °C; median={median:.2f} °C; Q3={q3:.2f} °C; IQR={iqr:.2f} °C")
print(f"Outlier fences: {lower_fence:.2f} to {upper_fence:.2f} °C")
print(f"Observed range: {temperature.min():.2f} to {temperature.max():.2f} °C")
print(f"Outlier count: {len(outliers)} ({len(outliers)/len(temperature):.2%})")
print("First 20 outlier values:", outliers.head(20).tolist())


Q1=19.46 °C; median=27.09 °C; Q3=34.50 °C; IQR=15.04 °C
Outlier fences: -3.10 to 57.06 °C
Observed range: 12.00 to 42.00 °C
Outlier count: 0 (0.00%)
First 20 outlier values: []


## 5. Grouped temperature frequency distribution

Ten equal-width classes follow the reference notebook. Each class is left closed and right open, except the final class, which includes the maximum. Grouped statistics below are **estimates** based on class midpoints and interpolation. The exact statistics above use the original observations.


In [5]:
class_count = 10
edges = np.linspace(temperature.min(), temperature.max(), class_count + 1)
counts, _ = np.histogram(temperature.to_numpy(), bins=edges)
midpoints = (edges[:-1] + edges[1:]) / 2
grouped = pd.DataFrame({"lower_C": edges[:-1], "upper_C": edges[1:],
                        "midpoint_C": midpoints, "frequency": counts})
grouped["cumulative_frequency"] = grouped["frequency"].cumsum()
print(grouped.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
assert int(grouped["frequency"].sum()) == len(temperature)


 lower_C  upper_C  midpoint_C  frequency  cumulative_frequency
  12.000   15.000      13.500        998                   998
  15.000   18.000      16.500       1007                  2005
  18.000   21.000      19.500       1021                  3026
  21.000   24.000      22.500        980                  4006
  24.000   27.000      25.500        968                  4974
  27.000   30.000      28.500        969                  5943
  30.000   33.000      31.500       1054                  6997
  33.000   36.000      34.500       1024                  8021
  36.000   39.000      37.500        999                  9020
  39.000   42.000      40.500        980                 10000


In [6]:
n = int(grouped["frequency"].sum())
width = edges[1] - edges[0]
frequencies = counts.astype(float)
grouped_mean = float(np.sum(frequencies * midpoints) / n)
grouped_variance = float(np.sum(frequencies * (midpoints - grouped_mean) ** 2) / (n - 1))

def grouped_quantile(fraction):
    target = fraction * n
    index = int(np.searchsorted(grouped["cumulative_frequency"].to_numpy(), target, side="left"))
    preceding = 0 if index == 0 else int(grouped["cumulative_frequency"].iloc[index - 1])
    return float(edges[index] + ((target - preceding) / counts[index]) * width)

modal_index = int(np.argmax(counts))
f_modal = counts[modal_index]
f_before = counts[modal_index - 1] if modal_index > 0 else 0
f_after = counts[modal_index + 1] if modal_index + 1 < class_count else 0
grouped_mode = float(edges[modal_index] +
                     ((f_modal - f_before) / (2 * f_modal - f_before - f_after)) * width)
gq1, gmedian, gq3 = (grouped_quantile(p) for p in (0.25, 0.50, 0.75))
grouped_results = {"frequency": n, "mean": grouped_mean, "mode": grouped_mode,
                   "median": gmedian, "variance": grouped_variance,
                   "std_dev": grouped_variance ** 0.5, "Q1": gq1,
                   "Q3": gq3, "IQR": gq3 - gq1}
print("Grouped estimates (temperature in °C):")
for name, value in grouped_results.items():
    print(f"{name}: {value:.3f}" if name != "frequency" else f"{name}: {value}")
print("\nExact versus grouped estimates:")
for name, exact in {"mean": temperature.mean(), "median": temperature.median(),
                    "variance": temperature.var(ddof=1), "std_dev": temperature.std(ddof=1),
                    "IQR": iqr}.items():
    print(f"{name}: exact={exact:.3f}, grouped={grouped_results[name]:.3f}")


Grouped estimates (temperature in °C):
frequency: 10000
mean: 27.003
mode: 32.217
median: 27.080
variance: 74.230
std_dev: 8.616
Q1: 19.454
Q3: 34.474
IQR: 15.019

Exact versus grouped estimates:
mean: exact=26.991, grouped=27.003
median: exact=27.090, grouped=27.080
variance: exact=75.066, grouped=74.230
std_dev: exact=8.664, grouped=8.616
IQR: exact=15.040, grouped=15.019
